# Memoria Semántica en Agentes de IA

## ¿Qué es la Memoria Semántica?

La **memoria semántica** permite a un agente **recordar hechos, preferencias y contexto sobre los usuarios** a través de múltiples conversaciones.

A diferencia de la memoria de conversación (que solo dura dentro de un mismo hilo), la memoria semántica:
- Persiste entre **diferentes sesiones** (`thread_id` distintos)
- Almacena información como **vectores** para búsqueda por similitud semántica
- Permite al agente recuperar recuerdos **relevantes al contexto actual**

En este notebook veremos:
1. El problema: un agente sin memoria no recuerda entre hilos
2. Solución: agregar herramientas de memoria semántica con `langmem`
3. Memorias por usuario: namespaces separados por `user_id`
4. Recuperación proactiva (eager): cargar memorias antes del LLM

---
## Paso 1: Instalación de dependencias

- **`langmem`**: Herramientas de memoria para agentes (manage, search)
- **`langgraph`**: Framework de agentes con grafos
- **`langchain-openai`**: Integración con modelos y embeddings de OpenAI
- **`python-dotenv`**: Para cargar variables de entorno desde `.env`

In [1]:
%%capture cap --no-stderr
%pip install -U langmem langgraph langchain-openai python-dotenv

---
## Paso 2: Configurar API Keys

El archivo `.env` en esta carpeta debe contener:
```
OPENAI_API_KEY=sk-proj-...
```

OpenAI se usa tanto para el **modelo de lenguaje** (`gpt-4o-mini`) como para los **embeddings** (`text-embedding-3-small`) que permiten la búsqueda semántica.

In [2]:
import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(dotenv_path=Path().resolve() / ".env", override=True)

# Desactivar LangSmith tracing para evitar errores 401
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ.pop("LANGCHAIN_API_KEY", None)
os.environ.pop("LANGSMITH_API_KEY", None)
os.environ.pop("OPENAI_BASE_URL", None)

key = os.environ.get("OPENAI_API_KEY", "")
if key:
    print(f"OPENAI_API_KEY cargada: {key[:8]}... (longitud: {len(key)})")
else:
    print("ERROR: No se encontró OPENAI_API_KEY en el .env")

OPENAI_API_KEY cargada: sk-proj-... (longitud: 164)


---
# Parte 1: El problema — Agente sin memoria entre hilos

## Paso 3: Crear los componentes base

Antes de agregar memoria semántica, veamos el problema que resuelve.

- **`InMemorySaver` (checkpointer)**: Guarda el historial de mensajes dentro de un mismo `thread_id`. Es la "memoria de corto plazo" del agente.
- **`InMemoryStore`**: Almacén de datos compartido entre hilos. Aquí es donde vivirán las memorias semánticas.

Sin embargo, en este primer ejemplo el store está vacío — el agente no tiene forma de guardar ni buscar recuerdos.

In [3]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langchain_openai import ChatOpenAI

checkpointer = InMemorySaver()  # Memoria de corto plazo (dentro de un hilo)
store = InMemoryStore()          # Almacén compartido (sin índice semántico aún)

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent = create_react_agent(model, tools=[], store=store, checkpointer=checkpointer)

c:\Users\Lenovo\anaconda3\envs\rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_14444\3992742444.py:10: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model, tools=[], store=store, checkpointer=checkpointer)


---
## Paso 4: Función helper `chat`

Creamos una función auxiliar que simplifica invocar al agente.

El parámetro `thread_id` es clave: actúa como un identificador de sesión. Conversaciones con el **mismo** `thread_id` comparten historial; conversaciones con **distinto** `thread_id` empiezan desde cero (sin memoria semántica).

In [4]:
def chat(agent, txt, thread_id, user_id=None):
    config = {"configurable": {"thread_id": thread_id}}
    if user_id:
        config["configurable"]["user_id"] = user_id
    result_state = agent.invoke(
        {"messages": [{"role": "user", "content": txt}]},
        config=config
    )
    return result_state["messages"][-1].content

---
## Paso 5: Conversación en hilo 1

El usuario pide un plan de entrenamiento. El agente responde dentro del `thread-1`.

In [5]:
thread_1 = "thread-1"
print(chat(agent, "Hi there, I'm training for a half marathon in 2 months - could you propose a daily training plan to prepare?", thread_1))

Absolutely! Training for a half marathon is an exciting challenge. Here’s a 2-month training plan that gradually builds your endurance and prepares you for race day. This plan assumes you have a base fitness level and can comfortably run at least 3 miles. Adjust the mileage according to your current fitness level if needed.

### Weekly Structure:
- **Monday**: Rest or cross-training
- **Tuesday**: Short run
- **Wednesday**: Speed work or hill training
- **Thursday**: Short run
- **Friday**: Rest or cross-training
- **Saturday**: Long run
- **Sunday**: Recovery run or rest

### Training Plan:

#### Week 1:
- **Monday**: Rest or cross-training (cycling, swimming, yoga)
- **Tuesday**: 3 miles easy
- **Wednesday**: 4 x 400m at 5K pace with 400m recovery jogs
- **Thursday**: 3 miles easy
- **Friday**: Rest or cross-training
- **Saturday**: 5 miles long run
- **Sunday**: 2 miles recovery run

#### Week 2:
- **Monday**: Rest or cross-training
- **Tuesday**: 3 miles easy
- **Wednesday**: 5 x 4

---
## Paso 6: Continuación en el mismo hilo

Dentro del **mismo hilo**, el agente recuerda todo gracias al `checkpointer`. El usuario dice "¡Buena suerte!" y el agente entiende el contexto.

In [6]:
print(chat(agent, "Nice! Wish me luck!", thread_1))

Good luck with your training and the half marathon! You've got this! Remember to enjoy the process and celebrate your progress along the way. If you have any questions or need further advice, feel free to ask. Happy running! 🏃‍♂️🏅


---
## Paso 7: El problema — hilo diferente, sin memoria

Ahora el **mismo usuario** inicia un nuevo hilo (`thread-2`). Sin memoria semántica, el agente no sabe nada sobre el entrenamiento previo.

La respuesta será genérica, sin contexto. Esto es el problema que vamos a resolver.

In [7]:
thread_2 = "thread-2"
print(chat(agent, "Nice! Oh thank you! It'll be hard.", thread_2))
# El agente no sabe a qué se refiere — no tiene contexto del hilo anterior

You're welcome! It sounds like you might be facing a challenge. If you want to share more about what you're dealing with, I'm here to help or offer support!


---
# Parte 2: Solución — Memoria Semántica con langmem

## Paso 8: Herramientas de memoria

Aquí está la solución. Usamos dos herramientas de `langmem`:

| Herramienta | Función |
|-------------|----------|
| `create_manage_memory_tool` | Permite al agente **guardar y actualizar** memorias |
| `create_search_memory_tool` | Permite al agente **buscar** memorias por similitud semántica |

### ¿Qué es el índice vectorial?
El `InMemoryStore` ahora tiene un **índice vectorial**:
- `dims: 1536` → dimensiones del vector (tamaño del embedding de OpenAI)
- `embed: "openai:text-embedding-3-small"` → modelo que convierte texto en vectores numéricos

Cuando el agente guarda una memoria, el texto se convierte en un vector. Cuando busca, convierte la consulta en vector y encuentra los recuerdos más similares (búsqueda semántica).

In [8]:
from langmem import create_manage_memory_tool, create_search_memory_tool
from langchain_openai import ChatOpenAI

# Store con índice vectorial para búsqueda semántica
store = InMemoryStore(
    index={
        "dims": 1536,
        "embed": "openai:text-embedding-3-small"
    }
)

namespace = ("agent_memories",)  # Espacio de nombres donde se guardan los recuerdos
memory_tools = [
    create_manage_memory_tool(namespace),  # Guardar/actualizar memorias
    create_search_memory_tool(namespace)   # Buscar memorias
]
checkpointer = InMemorySaver()

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent = create_react_agent(model, tools=memory_tools, store=store, checkpointer=checkpointer)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_14444\4145477246.py:20: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model, tools=memory_tools, store=store, checkpointer=checkpointer)


---
## Paso 9: Primera conversación — el agente guarda memorias

El usuario pide el plan y luego explícitamente pide que se guarden los recuerdos. El agente usará `manage_memory_tool` para almacenar el plan en el store vectorial.

In [9]:
thread_1 = "thread-1"
print(chat(agent, "Hi there, I'm training for a half marathon in 2 months - could you propose a daily training plan to prepare?", thread_1))

Here's a daily training plan to help you prepare for your half marathon in 2 months. This plan assumes you have a base level of fitness and can comfortably run at least 3 miles. Adjust the distances and paces according to your current fitness level.

### Weekly Training Plan

**Week 1-2: Building Base**
- **Monday:** 3 miles easy run
- **Tuesday:** Cross-training (cycling, swimming, or strength training)
- **Wednesday:** 4 miles easy run
- **Thursday:** Rest or light yoga/stretching
- **Friday:** 3 miles tempo run (run at a challenging but sustainable pace)
- **Saturday:** Long run (5 miles)
- **Sunday:** Rest or active recovery (walking, light jogging)

**Week 3-4: Increasing Mileage**
- **Monday:** 4 miles easy run
- **Tuesday:** Cross-training
- **Wednesday:** 5 miles easy run
- **Thursday:** Rest or light yoga/stretching
- **Friday:** 4 miles tempo run
- **Saturday:** Long run (6-7 miles)
- **Sunday:** Rest or active recovery

**Week 5-6: Peak Training**
- **Monday:** 5 miles easy 

In [10]:
# El usuario pide explícitamente guardar los recuerdos
print(chat(agent, "Nice! Wish me luck! Please note down the detailed memories for me :)", thread_1))

Good luck with your training! I've noted down all the details of your half marathon training plan. If you need any more assistance or adjustments along the way, feel free to ask!


---
## Paso 10: Segundo hilo — el agente recuerda gracias a la búsqueda semántica

Ahora abrimos un `thread-2` completamente nuevo. El agente no tiene historial de conversación con este hilo, pero puede **buscar en el store** para encontrar memorias relevantes.

El agente convierte la pregunta en un vector, busca memorias similares, y encuentra el plan de entrenamiento guardado antes.

In [11]:
thread_2 = "thread-2"
print(chat(agent, "Remember what I'm supposed to do for my training this week? It's week 3...", thread_2))

For week 3 of your training, you should follow this plan:

- **4 miles easy run**
- **Cross-training**
- **5 miles easy run**
- **Rest/light yoga**
- **4 miles tempo run**
- **Long run (6-7 miles)**
- **Rest/active recovery**

You're training for a half marathon in 2 months, so make sure to stick to this schedule!


---
## Paso 11: Actualizar memorias existentes

El usuario tuvo un accidente. Le pide al agente que **actualice** el plan existente (no crear uno nuevo). El agente usa `manage_memory_tool` con la clave existente para modificar el recuerdo en el store.

Esto demuestra que las memorias no son inmutables — pueden evolucionar con el tiempo.

In [12]:
print(chat(agent, "That may be tricky. I just sprained my ankle. Could you update my plan to include more cross training? Be sure to update the existing key of our plan", thread_2))

I've updated your training plan to include more emphasis on low-impact cross-training due to your sprained ankle. Here's the revised plan for week 3:

- **4 miles easy run**
- **Cross-training (more emphasis on low-impact activities)**
- **5 miles easy run**
- **Rest/light yoga**
- **4 miles tempo run**
- **Long run (6-7 miles)**
- **Rest/active recovery**

Take care of your ankle, and adjust the activities as needed!


---
# Parte 3: Memorias por usuario

## Paso 12: Namespace dinámico con `{user_id}`

El problema anterior guardaba todo en el mismo espacio: `("agent_memories",)`. Esto es un problema en una app real con múltiples usuarios: **todos compartirían las mismas memorias**.

La solución es usar un namespace dinámico: `("agent_memories", "{user_id}")`. LangGraph reemplaza `{user_id}` automáticamente con el valor del configurable, creando un espacio de nombres separado por usuario.

También actualizamos `chat()` para recibir `user_id` y pasarlo en la configuración.

In [13]:
from langmem import create_manage_memory_tool, create_search_memory_tool
from langchain_openai import ChatOpenAI

store = InMemoryStore(
    index={
        "dims": 1536,
        "embed": "openai:text-embedding-3-small"
    }
)

# {user_id} se reemplaza dinámicamente → cada usuario tiene su propio espacio
namespace = ("agent_memories", "{user_id}")
memory_tools = [
    create_manage_memory_tool(namespace),
    create_search_memory_tool(namespace)
]
checkpointer = InMemorySaver()

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent = create_react_agent(model, tools=memory_tools, store=store, checkpointer=checkpointer)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_14444\820647983.py:20: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model, tools=memory_tools, store=store, checkpointer=checkpointer)


---
## Paso 13: Usuario A — Will, entrenamiento para maratón

El usuario A (Will) pide un plan de entrenamiento. Sus memorias se guardarán en el namespace `("agent_memories", "User-A")`.

In [14]:
thread_1 = "thread-1"
user_id_a = "User-A"

print(chat(
    agent,
    "Hi I'm Will, I'm training for a half marathon in 2 months - could you propose a daily training plan to prepare and help me stay honest??",
    thread_1,
    user_id_a
))

Hi Will! That's an exciting goal! Here's a daily training plan to help you prepare for your half marathon over the next two months. This plan assumes you have a base level of fitness and can comfortably run at least 3 miles. 

### Weekly Training Plan

**Week 1-2: Building Base**
- **Monday:** Rest or cross-training (cycling, swimming, yoga)
- **Tuesday:** 3 miles easy run
- **Wednesday:** 4 miles at a moderate pace
- **Thursday:** Rest or cross-training
- **Friday:** 3 miles easy run
- **Saturday:** Long run (5 miles)
- **Sunday:** Rest

**Week 3-4: Increasing Mileage**
- **Monday:** Rest or cross-training
- **Tuesday:** 4 miles easy run
- **Wednesday:** 5 miles at a moderate pace
- **Thursday:** Rest or cross-training
- **Friday:** 4 miles easy run
- **Saturday:** Long run (6-7 miles)
- **Sunday:** Rest

**Week 5-6: Building Endurance**
- **Monday:** Rest or cross-training
- **Tuesday:** 5 miles easy run
- **Wednesday:** 6 miles at a moderate pace
- **Thursday:** Rest or cross-traini

---
## Paso 14: Usuario B — John, aprendiendo ajedrez

El usuario B (John) tiene un interés completamente diferente. Sus memorias se guardan en `("agent_memories", "User-B")`, **separadas** de las de Will.

In [15]:
thread_2 = "thread-2"
user_id_b = "User-B"

print(chat(
    agent,
    "Hi I'm John, I'm learning chess - could you help me become great??",
    thread_2,
    user_id_b
))

Hi John! I'd love to help you become great at chess. What specific areas are you looking to improve in? For example, are you interested in learning openings, tactics, endgames, or general strategies?


---
## Paso 15: Verificar el aislamiento de memorias entre usuarios

Le preguntamos al agente (en el contexto de User-B) si recuerda deportes. El agente solo ve las memorias de John (ajedrez), **no** las de Will (maratón).

Esto demuestra el aislamiento de memorias por usuario.

In [16]:
print(chat(
    agent,
    "Do you remember me liking any sports?",
    thread_2,
    user_id_b
))

I don't have any information about you liking any sports in my memory. If you have any favorite sports or activities, feel free to share!


---
## Paso 16: Inspeccionar el store — ver todas las memorias guardadas

`store.search()` permite ver qué hay almacenado. El resultado muestra:
- El **namespace** de cada memoria (identifica a qué usuario pertenece)
- El **valor** guardado (texto de la memoria)

Podemos confirmar que Will y John tienen memorias completamente separadas.

In [17]:
# Ver todas las memorias en el store
items = store.search(("agent_memories",))
for item in items:
    print(f"Namespace: {item.namespace}")
    print(f"Contenido: {item.value}")
    print("-" * 50)

Namespace: ('agent_memories', 'User-A')
Contenido: {'content': "User's name is Will and he is training for a half marathon in 2 months."}
--------------------------------------------------
Namespace: ('agent_memories', 'User-B')
Contenido: {'content': "User's name is John and he is learning chess."}
--------------------------------------------------


---
# Parte 4: Recuperación Proactiva (Eager Memory Retrieval)

## Paso 17: ¿Cuál es el problema de la búsqueda reactiva?

En el enfoque anterior, el agente **decide cuándo buscar** en el store. Esto tiene un problema:

- El agente no sabe qué no sabe ("unknown unknowns")
- Ante preguntas ambiguas, a veces no busca en memoria y responde genéricamente
- Requiere que el usuario haga preguntas muy explícitas

La solución es la **recuperación proactiva (eager)**: buscar memorias relevantes **antes** de la primera llamada al LLM, e inyectarlas en el sistema como contexto.

## Paso 18: Agente con función `prompt` y recuperación proactiva

La función `prompt` se ejecuta antes de cada llamada al LLM:
1. Toma el último mensaje del usuario como consulta
2. Busca en el store memorias semánticamente similares
3. Las inyecta en el mensaje de sistema

Así, el LLM **siempre tiene el contexto relevante disponible** sin necesidad de decidir si buscar o no.

In [18]:
from langmem import create_manage_memory_tool, create_search_memory_tool
from langgraph.config import get_store
from langchain_openai import ChatOpenAI

store = InMemoryStore(
    index={
        "dims": 1536,
        "embed": "openai:text-embedding-3-small"
    }
)

namespace = ("agent_memories",)
memory_tools = [
    create_manage_memory_tool(namespace),
    create_search_memory_tool(namespace)
]
checkpointer = InMemorySaver()

def prompt(state):
    # Obtener el store desde el contexto de LangGraph
    store = get_store()
    # Buscar memorias relevantes usando el último mensaje como consulta
    items = store.search(namespace, query=state["messages"][-1].content)
    # Formatear los recuerdos encontrados
    memories = "\n\n".join(str(item) for item in items)
    # Inyectar memorias en el mensaje de sistema
    system_msg = {"role": "system", "content": f"## Memories:\n\n{memories}"}
    return [system_msg] + state["messages"]

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent = create_react_agent(
    model,
    prompt=prompt,          # Se ejecuta ANTES de cada llamada al LLM
    tools=memory_tools,
    store=store,
    checkpointer=checkpointer
)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_14444\3504170796.py:31: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


---
## Paso 19: Guardar memorias en hilo 1

El usuario pide el plan y pide guardarlo. El agente usa `manage_memory_tool` para almacenar los detalles.

In [19]:
thread_1 = "thread-1"
print(chat(agent, "Hi there, I'm training for a half marathon in 2 months - could you propose a daily training plan to prepare?", thread_1))

Here's a daily training plan to help you prepare for your half marathon in 2 months. This plan assumes you have a basic level of fitness and can run at least 3 miles comfortably. Adjust the distances and paces according to your current fitness level.

### Weekly Training Plan

**Week 1-2: Building Base**
- **Monday:** Rest or cross-training (cycling, swimming, yoga)
- **Tuesday:** 3 miles easy run
- **Wednesday:** 4 miles at a comfortable pace
- **Thursday:** Rest or cross-training
- **Friday:** 3 miles easy run
- **Saturday:** Long run (5 miles)
- **Sunday:** Rest or light activity (walking, stretching)

**Week 3-4: Increasing Mileage**
- **Monday:** Rest or cross-training
- **Tuesday:** 4 miles easy run
- **Wednesday:** 5 miles at a comfortable pace
- **Thursday:** Rest or cross-training
- **Friday:** 4 miles easy run
- **Saturday:** Long run (6-7 miles)
- **Sunday:** Rest or light activity

**Week 5-6: Speed and Endurance**
- **Monday:** Rest or cross-training
- **Tuesday:** 5 miles

In [20]:
print(chat(agent, "Nice! Wish me luck! Please note down the detailed memories for me :)", thread_1))

I've noted down the detailed training plan for your half marathon. Best of luck with your training! If you need any more assistance or adjustments along the way, feel free to ask. You've got this! 🏃‍♂️💪


---
## Paso 20: Recuperación proactiva en hilo 2

En el nuevo hilo, el agente busca memorias automáticamente **antes** del LLM usando la función `prompt`:
1. Convierte "training this week? It's week 3" en un vector
2. Busca en el store y encuentra el plan de entrenamiento guardado
3. Lo inyecta como contexto del sistema
4. El LLM responde directamente con la información correcta, sin necesidad de invocar herramientas

Esto es más eficiente: **menos llamadas al LLM**, respuestas más rápidas y directas.

In [21]:
thread_2 = "thread-2"
print(chat(agent, "What I'm supposed to do for my training this week? It's week 3...", thread_2))

For week 3 of your half marathon training, your plan is as follows:

- **Monday**: Rest/cross-training
- **Tuesday**: 4 miles easy run
- **Wednesday**: 5 miles
- **Thursday**: Rest/cross-training
- **Friday**: 4 miles easy run
- **Saturday**: Long run (6-7 miles)
- **Sunday**: Rest/light activity

Make sure to listen to your body and adjust as needed!


---
# Resumen: ¿Qué aprendimos?

## Comparación de enfoques

| Enfoque | Persistencia | Búsqueda | Cuándo usar |
|---------|-------------|----------|-------------|
| Sin memoria | Solo en el hilo | No | Chat simple sin contexto |
| Memoria semántica reactiva | Entre hilos | El agente decide cuándo buscar | Usuarios que piden explícitamente recordar |
| Memoria por usuario | Entre hilos, aislada | El agente decide | Apps multi-usuario |
| Recuperación proactiva (eager) | Entre hilos | Automática antes del LLM | Apps donde el contexto siempre importa |

## Conceptos clave

| Concepto | Descripción |
|----------|-------------|
| **InMemoryStore con índice** | Almacén vectorial que permite búsqueda por similitud semántica |
| **Embeddings** | Representaciones numéricas del texto que capturan su significado |
| **`create_manage_memory_tool`** | Permite al agente guardar/actualizar memorias como herramienta |
| **`create_search_memory_tool`** | Permite al agente buscar memorias relevantes como herramienta |
| **Namespace con `{user_id}`** | Aísla memorias por usuario automáticamente |
| **Función `prompt` proactiva** | Inyecta memorias relevantes antes de cada llamada al LLM |

## Flujo de memoria semántica

```
Conversación (hilo 1)
  → Agente guarda memorias en store vectorial
  
Nueva conversación (hilo 2)
  → prompt() busca memorias relevantes
  → LLM recibe contexto enriquecido
  → Responde como si "recordara"
```